[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/07b_continued_pretraining.ipynb)

# Continued pre-training — domain adaptation (self-supervised)

> **Fine-tuning series — 07b of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.


## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos días."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' — a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## Continued pre-training (domain adaptation)  ·  *what-axis*

The **self-supervised** half of fine-tuning that notebook 02 set up but never ran. Instead of
(instruction → response) pairs, you keep training the **next-token objective on raw domain
text** — no labels, no chat template, loss on *every* token. Also called *domain-adaptive
pre-training* (DAPT). Use it to push domain knowledge, vocabulary, or a writing style **into the
weights** (medical, legal, code, a new language) before you instruction-tune on top.

Pipeline: **continued pre-training (base model) → instruction tuning → preference optimization.**
Reach for it when the knowledge must live in the weights or you're adapting vocabulary/style;
if you just need *lookup* facts, RAG is usually cheaper.

### How it differs from SFT

| | SFT / instruction tuning | Continued pre-training |
|---|---|---|
| Data | prompt → response pairs | raw text (documents) |
| Loss masked to response? | yes | **no — all tokens** |
| Start from | usually an *instruct* model | usually a **base** model |
| Learning rate | ~2e-4 (LoRA) | **lower**, ~5e-5, more data, longer |
| Goal | behavior / format | **knowledge / vocabulary / style** |

### Prepare raw text (tokenize → pack into fixed blocks)

No chat template. Concatenate documents and chop into equal-length blocks — the classic causal-LM
packing so no compute is wasted on padding.

In [ ]:
from datasets import Dataset
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

raw = Dataset.from_dict({"text": [          # your raw domain documents go here
    "The myocardium is the muscular middle layer of the heart wall. ",
    "Ejection fraction measures the percentage of blood pumped per beat. ",
    "B-lines on lung ultrasound indicate interstitial syndrome. ",
]})

block = 128
def tok(b):  return tokenizer(b["text"])
def pack(b):
    ids = sum(b["input_ids"], [])
    n = (len(ids) // block) * block
    return {"input_ids": [ids[i:i+block] for i in range(0, n, block)]}

lm_ds = raw.map(tok, batched=True, remove_columns=raw.column_names).map(pack, batched=True)
print(lm_ds)

### Train (next-token on all tokens)

A plain `Trainer` with `DataCollatorForLanguageModeling(mlm=False)` builds the shifted labels for
you. Wrap the model with `get_peft_model(model, lora_cfg)` first for a cheap LoRA domain-adapt, or
train it fully for deeper knowledge injection.

In [ ]:
import torch
from transformers import (AutoModelForCausalLM, Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling)

model = AutoModelForCausalLM.from_pretrained(           # a BASE model is the usual choice here
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)   # causal LM, not masked LM

trainer = Trainer(model=model,
    args=TrainingArguments(output_dir="cpt", per_device_train_batch_size=2,
        max_steps=20, learning_rate=5e-5, bf16=BF16_OK, logging_steps=5, report_to="none"),
    train_dataset=lm_ds, data_collator=collator)
trainer.train()
del trainer, model; cleanup()

### Tips

- **Mix in some general text** (a few % of generic data) so the model doesn't forget its base
  abilities — continued pre-training is the most forgetting-prone setup.
- **Then instruction-tune** (notebook 08) on top to make the domain knowledge usable in chat.
- Adding new tokens/a new language? **Unsloth's continued-pre-training path** also trains the
  `embed_tokens` and `lm_head` (often at a separate, smaller LR) — worth it when the tokenizer
  doesn't cover your domain well.